In [2]:
# !pip install sentence-transformers

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
df = pd.read_csv(r"C:\Users\User\Downloads\Analysis on Music streaming platforms\Data\processed_data\survey_data_cleaned_new.csv")

# Basic cleaning (minimal since already cleaned)
df = df.dropna(subset=['clean_text'])
df = df[df['clean_text'].str.strip() != ""]

df.reset_index(drop=True, inplace=True)

KeyError: ['clean_text']

In [26]:
def is_hindi(text):
    return any('\u0900' <= ch <= '\u097F' for ch in text)

df['is_hindi'] = df['clean_text'].apply(is_hindi)

In [27]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['clean_text'].apply(get_sentiment)

In [28]:
def is_meaningful(text):
    return len(text.split()) > 3

df = df[df['clean_text'].apply(is_meaningful)]

In [29]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df['clean_text'].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [30]:
k = 5  # keep simple

kmeans = KMeans(n_clusters=k, random_state=42)
df['cluster'] = kmeans.fit_predict(embeddings)

C:\Users\User\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


In [31]:
def get_examples(embeddings, labels, texts, n=5):
    result = {}
    
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        cluster_emb = embeddings[idx]
        
        centroid = cluster_emb.mean(axis=0).reshape(1, -1)
        sim = cosine_similarity(cluster_emb, centroid).flatten()
        
        top_idx = sim.argsort()[-n:]
        result[c] = [texts[i] for i in idx[top_idx]]
    
    return result

cluster_examples = get_examples(
    embeddings,
    df['cluster'].values,
    df['clean_text'].values
)

for c, texts in cluster_examples.items():
    print(f"\nCluster {c}:")
    for t in texts:
        print("-", t)


Cluster 0:
- worst app ever seen it can t provide songs search
- totally useless app can t listen to any song without money
- one of the worst music app which doesn t have proper structure and no user experience even unable to search the song that is too irritating please kindly update the app and fix the issues otherwise it will be useless for me someone
- very bad app without premium subscription you can t here atleast 01 song only add is on nothing else
- this is very bad app because all songs are not available in this app

Cluster 1:
- i love how i can find all musics i want
- daily playlist suggestions are amazing i discover new music every day tailored perfectly for me
- great platform to hear music but a bit disappointing that we need to take premium to listen to break free and loopful music but overall performance is amazing
- used to be good i can t discover new songs that i might like because all the mixes radios etc are basically all the same songs spotify keeps on recommen

In [37]:
def assign_topic(text):
    
    # Ads
    if 'ads' in text:
        return "Ads Issues"
    
    # Premium
    if 'premium' in text or 'subscription' in text:
        return "Premium Issues"
    
    # Performance
    if any(word in text for word in ['crash', 'lag', 'slow']):
        return "Performance Issues"
    
    # Content/Search
    if any(word in text for word in ['search', 'not available', 'missing']):
        return "Content/Search Issues"
    
    # Recommendation / Personalization
    if any(word in text for word in ['recommend', 'suggest', 'discover']):
        return "Recommendation Issues"
    
    # User Experience 
    if any(word in text for word in ['easy', 'interface', 'experience', 'freedom']):
        return "User Experience"
    
    return "General Feedback"

In [42]:
def refine_sentiment(text, base):
    
    if 'but' in text or 'however' in text:
        parts = text.split('but') if 'but' in text else text.split('however')
        if len(parts) > 1:
            return get_sentiment(parts[-1])
    
    return base

In [43]:
df.loc[df['is_hindi'] == True, 'topic'] = "Regional Feedback"

In [44]:
df = df.dropna(subset=['topic', 'sentiment'])
df.reset_index(drop=True, inplace=True)

In [45]:
summary = df.groupby(['topic', 'sentiment'])['clean_text'] \
    .count().reset_index()

summary.columns = ['topic', 'sentiment', 'count']

In [46]:
final_df = df[['app_name', 'rating', 'clean_text', 'cluster', 'topic', 'sentiment']]

final_df.to_csv("final_output.csv", index=False)
summary.to_csv("summary_output.csv", index=False)

print(final_df.head())

  app_name  rating                                         clean_text  \
0  Spotify       5  just like 80 of black women i to have had the ...   
1  Spotify       4  i like the fact they give you more freedom to ...   
2  Spotify       4  it s a great app however the reason for writin...   
3  Spotify       5  i play music everyday on this app and it s get...   
4  Spotify       5  spotify music is the best music platform i hav...   

   cluster                  topic sentiment  
0        2       General Feedback  Positive  
1        1       General Feedback  Positive  
2        4  Content/Search Issues  Positive  
3        0       General Feedback  Positive  
4        1       General Feedback   Neutral  
